In [5]:
import pandas as pd
from pathlib import Path

DATASET_PATH = Path('datasets/ecommerce-events-history-in-electronics-store')

In [7]:
df = pd.read_csv(
    DATASET_PATH / 'events.csv',
    parse_dates=['event_time'],
    date_format='%Y-%m-%d %H:%M:%S UTC',
    dtype={
        'event_type': 'category',
        'product_id': 'int32',
        'category_id': 'int64',
        'category_code': 'category',
        'brand': 'category',
        'price': 'float32',
        'user_id': 'int32',
        'user_session': 'string'
    }
)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 885129 entries, 0 to 885128
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   event_time     885129 non-null  datetime64[ns]
 1   event_type     885129 non-null  category      
 2   product_id     885129 non-null  int32         
 3   category_id    885129 non-null  int64         
 4   category_code  648910 non-null  category      
 5   brand          672765 non-null  category      
 6   price          885129 non-null  float32       
 7   user_id        885129 non-null  int32         
 8   user_session   884964 non-null  string        
dtypes: category(3), datetime64[ns](1), float32(1), int32(2), int64(1), string(1)
memory usage: 33.8 MB


In [8]:
df['user_session'].nunique()

490398

```
Размер таблицы: (885129, 9)
Количество уникальных product_id: 53453
Количество уникальных category_id: 718
Количество уникальных category_code: 107
Количество уникальных брендов: 999

---

Примеры product_id с более чем одной категорией:
Empty DataFrame
Columns: [product_id, num_category_codes]
Index: []

---

Примеры product_id с более чем одним кодом категории:
Empty DataFrame
Columns: [product_id, num_category_ids]
Index: []

---

Примеры product_id с более чем одним брендом:
Empty DataFrame
Columns: [product_id, num_brands]
Index: []
```

In [49]:
# 1. Распределение типов событий
print("Распределение event_type:")
print(df["event_type"].value_counts())
print("\n---\n")

# 2. Статистика цен
print("Статистика цены:")
print(df["price"].describe())
print("\n---\n")

# 3. Анализ сессий: длительность сессии
session_times = df.groupby("user_session")["event_time"].agg(["min", "max"])
session_times["session_duration_sec"] = (session_times["max"] - session_times["min"]).dt.total_seconds()
print("Статистика длительности сессий (секунды):")
print(session_times["session_duration_sec"].describe())
print("\n---\n")

# Анализ сессий: количество событий на сессию
session_counts = df["user_session"].value_counts()
print("Статистика количества событий на сессию:")
print(session_counts.describe())


Распределение event_type:
event_type
view        793748
cart         54035
purchase     37346
Name: count, dtype: int64

---

Статистика цены:
count    885129.000000
mean        146.328690
std         296.807678
min           0.220000
25%          26.459999
50%          65.709999
75%         190.490005
max       64771.058594
Name: price, dtype: float64

---

Статистика длительности сессий (секунды):
count    4.903980e+05
mean     2.522750e+04
std      2.987723e+05
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      3.000000e+01
max      1.339442e+07
Name: session_duration_sec, dtype: float64

---

Статистика количества событий на сессию:
count    490398.0
mean     1.804583
std      2.947467
min           1.0
25%           1.0
50%           1.0
75%           2.0
max         572.0
Name: count, dtype: Float64


In [51]:
import pandas as pd

# 1. Анализ конверсий: сначала определим количество событий в сессиях
session_event_counts = df.groupby("user_session")["event_type"].transform("count")
multi_session = df[session_event_counts > 1]

# Для каждого сеанса собираем список событий, затем вычисляем доли, где встречается 'cart' и 'purchase'
session_events = multi_session.groupby("user_session")["event_type"].agg(list).reset_index()
session_events["has_cart"] = session_events["event_type"].apply(lambda events: "cart" in events)
session_events["has_purchase"] = session_events["event_type"].apply(lambda events: "purchase" in events)

total_sessions = session_events.shape[0]
sessions_with_cart = session_events["has_cart"].sum()
sessions_with_purchase = session_events["has_purchase"].sum()

print("Количество сессий с более чем 1 событием:", total_sessions)
print("Доля сессий с событием cart:", sessions_with_cart / total_sessions)
print("Доля сессий с событием purchase:", sessions_with_purchase / total_sessions)
print("\n---\n")

# 2. Анализ переходов между событиями
# Сначала сортируем DataFrame по сессиям и времени события
df_sorted = df.sort_values(["user_session", "event_time"])
# Для каждого события находим следующее событие в той же сессии
df_sorted["next_event"] = df_sorted.groupby("user_session")["event_type"].shift(-1)
# Отбрасываем строки, где нет следующего события
transitions = df_sorted.dropna(subset=["next_event"])
# Группируем по парам событий и считаем их количество
transition_counts = transitions.groupby(["event_type", "next_event"]).size().reset_index(name="count")
print("Переходы между событиями:")
print(transition_counts)


Количество сессий с более чем 1 событием: 140731
Доля сессий с событием cart: 0.29321187229537204
Доля сессий с событием purchase: 0.1609524553936233

---

Переходы между событиями:
  event_type next_event   count
0       cart       cart     681
1       cart   purchase   20739
2       cart       view   20750
3   purchase       cart      94
4   purchase   purchase    9109
5   purchase       view   11678
6       view       cart   53202
7       view   purchase    5011
8       view       view  273302


/tmp/ipykernel_32470/1936230343.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  transition_counts = transitions.groupby(["event_type", "next_event"]).size().reset_index(name="count")


In [52]:
# 1. Анализ интервалов между событиями в сессии
df_sorted = df.sort_values(["user_session", "event_time"])
df_sorted["time_diff_sec"] = df_sorted.groupby("user_session")["event_time"].diff().dt.total_seconds()
print("Статистика интервалов между событиями (в секундах):")
print(df_sorted["time_diff_sec"].describe())
print("\n---\n")

# 2. Средняя цена по типам событий
price_by_event = df.groupby("event_type")["price"].mean().reset_index()
print("Средняя цена по типам событий:")
print(price_by_event)


Статистика интервалов между событиями (в секундах):
count    3.945660e+05
mean     3.135474e+04
std      2.776168e+05
min      0.000000e+00
25%      2.800000e+01
50%      8.000000e+01
75%      3.120000e+02
max      1.233801e+07
Name: time_diff_sec, dtype: float64

---

Средняя цена по типам событий:
  event_type       price
0       cart  159.637512
1   purchase  137.240814
2       view  145.850296


/tmp/ipykernel_32470/2452002502.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  price_by_event = df.groupby("event_type")["price"].mean().reset_index()


In [53]:
from collections import Counter

def extract_ngrams(events, n):
    """Возвращает список n-грамм из списка событий."""
    return [tuple(events[i:i+n]) for i in range(len(events)-n+1)]

# Группируем события по сессии, сортируя их по времени
session_events = df.sort_values(["user_session", "event_time"]).groupby("user_session")["event_type"].agg(list)

ngrams_purchase = Counter()
n = 3  # Анализируем последовательности из 3 событий

for events in session_events:
    ngrams = extract_ngrams(events, n)
    # Считаем только те последовательности, где последнее событие — 'purchase'
    for gram in ngrams:
        if gram[-1] == "purchase":
            ngrams_purchase[gram] += 1

print("Топ-10 триграмм (последовательностей из 3 событий), предшествующих покупке:")
for gram, count in ngrams_purchase.most_common(10):
    print(gram, count)

print("\n---\n")


Топ-10 триграмм (последовательностей из 3 событий), предшествующих покупке:
('view', 'cart', 'purchase') 20559
('cart', 'purchase', 'purchase') 3912
('purchase', 'purchase', 'purchase') 3545
('view', 'view', 'purchase') 1729
('cart', 'view', 'purchase') 1516
('view', 'purchase', 'purchase') 1234
('purchase', 'view', 'purchase') 980
('cart', 'cart', 'purchase') 111
('purchase', 'cart', 'purchase') 60

---



In [54]:
# Группировка по сессиям для расчёта основных метрик
session_stats = df.groupby("user_session").agg(
    session_start = ("event_time", "min"),
    session_end = ("event_time", "max"),
    event_count = ("event_time", "count")
)
# Вычисляем длительность сессии в секундах
session_stats["session_duration_sec"] = (session_stats["session_end"] - session_stats["session_start"]).dt.total_seconds()
# Среднее время между событиями
session_stats["avg_time_between"] = session_stats["session_duration_sec"] / (session_stats["event_count"] - 1)
# На случай деления на ноль
session_stats = session_stats.replace([float('inf'), -float('inf')], 0)

print("Статистика по сессиям:")
print(session_stats.describe())

print("\n---\n")


Статистика по сессиям:
                       session_start                    session_end  \
count                         490398                         490398   
mean   2020-12-12 06:35:07.978701056  2020-12-12 13:35:35.479467776   
min              2020-09-24 11:57:06            2020-09-24 11:57:06   
25%       2020-11-03 17:32:22.500000     2020-11-04 04:05:42.500000   
50%              2020-12-10 12:33:54            2020-12-10 19:35:42   
75%    2021-01-21 06:50:59.249999872  2021-01-21 12:14:17.249999872   
max              2021-02-28 23:58:14            2021-02-28 23:59:09   
std                              NaN                            NaN   

         event_count  session_duration_sec  avg_time_between  
count  490398.000000          4.903980e+05      1.407310e+05  
mean        1.804583          2.522750e+04      3.619241e+04  
min         1.000000          0.000000e+00      0.000000e+00  
25%         1.000000          0.000000e+00      4.400000e+01  
50%         1.000000  

In [55]:
# Средняя цена по категориям
price_by_category = df.groupby("category_code")["price"].agg(["mean", "count"]).reset_index()
print("Топ-10 категорий с самой высокой средней ценой:")
print(price_by_category.sort_values("mean", ascending=False).head(10))

print("\n---\n")

# Средняя цена по брендам
price_by_brand = df.groupby("brand")["price"].agg(["mean", "count"]).reset_index()
print("Топ-10 брендов с самой высокой средней ценой:")
print(price_by_brand.sort_values("mean", ascending=False).head(10))


Топ-10 категорий с самой высокой средней ценой:
                          category_code        mean   count
88          electronics.video.projector  746.783630    1587
20            appliances.kitchen.washer  666.669983      60
80  electronics.audio.music_tools.piano  436.800262     415
72              country_yard.cultivator  429.077728    2395
45      computers.components.videocards  385.903290  116717
54        computers.peripherals.monitor  375.537354    6688
89                 electronics.video.tv  354.135742   21398
96                          kids.skates  334.848022     196
9     appliances.kitchen.coffee_machine  331.044525    3454
83             electronics.camera.video  324.381409    1603

---

Топ-10 брендов с самой высокой средней ценой:
          brand         mean  count
871  infortrend  5523.655762      5
986      magner  1554.439941     16
5          achi  1464.750000     13
35        apple  1432.976196    856
34          apc  1296.115356    186
536   powerware  1171.31

/tmp/ipykernel_32470/3718371959.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  price_by_category = df.groupby("category_code")["price"].agg(["mean", "count"]).reset_index()
/tmp/ipykernel_32470/3718371959.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  price_by_brand = df.groupby("brand")["price"].agg(["mean", "count"]).reset_index()


### Постановка задачи

**Цель:**  
Разработать RL-агента, который, взаимодействуя с пользователями (или их имитацией), будет выбирать рекомендации таким образом, чтобы максимизировать общую ценность (например, конверсии, покупки или доход).

---

### Ключевые компоненты задачи

1. **Агент:**  
   - **Что делает:** Выбирает, какой товар или набор товаров рекомендовать пользователю на каждом шаге.  
   - **Методы обучения:** Можно использовать алгоритмы Q-learning, DQN, policy gradient или их гибридные варианты, а также рассмотреть offline RL, если основная информация доступна из исторических данных.

2. **Окружение:**  
   - **Что представляет:** Интерактивная система, в которой происходит взаимодействие с пользователями.  
   - **Характеристики:**  
     - Сессии пользователей, каждая из которых содержит историю действий (просмотры, добавления в корзину, покупки).  
     - Временные характеристики: время между событиями, длительность сессии и т.п.

3. **Состояние (State):**  
   - **Составляющие:**  
     - История сессии: последние n действий пользователя (например, последовательность событий «view», «cart», «purchase»).  
     - Контекст сессии: время начала сессии, длительность, активность пользователя.  
     - Информация о продуктах: цена, категория, бренд и другие характеристики.  
   - **Задача:** Сформировать представление, которое адекватно отражает текущую ситуацию пользователя для принятия рекомендаций.

4. **Действия (Action):**  
   - **Что выбирается:** Рекомендация конкретного товара или набора товаров для показа пользователю.  
   - **Особенности:**  
     - Действия могут быть дискретными (выбор из ограниченного набора товаров) или непрерывными (например, ранжирование товаров по релевантности).

5. **Награда (Reward):**  
   - **Параметры:**  
     - **Положительная награда:** При успешном переходе по этапу воронки, например:
       - +1 за клик или просмотр,
       - +2 за добавление в корзину,
       - +5 за совершение покупки (или даже значение, пропорциональное прибыли).  
     - **Отрицательная награда:** Если рекомендация не приводит к никакому взаимодействию или приводит к негативному исходу (например, отток пользователя).  
   - **Особенность:** Функция награды должна учитывать разницу между событиями: переход от просмотра к покупке может оцениваться существенно выше, чем просто просмотр.

6. **Переходы (Transitions):**  
   - **Как работают:** Состояние сессии обновляется на основании действия агента и реакции пользователя (на основе исторических данных или модели поведения).  
   - **Временная динамика:** Временные интервалы между событиями могут быть включены в описание состояния, что поможет учитывать задержки между действиями.

---

### Задача RL

**Формализация:**  
Найти оптимальную политику $ \pi(s) $, которая, для каждого состояния $ s $ (описание сессии и контекста пользователя), выбирает действие $ a $ (рекомендацию товара), максимизируя ожидаемую суммарную награду $ \mathbb{E}\left[\sum_{t=0}^{T} \gamma^t r_t\right] $, где $ r_t $ — награда на шаге $ t $, а $ \gamma $ — коэффициент дисконтирования.

**Особенности для рекомендательной системы:**
- **Эксплорация и эксплуатация:** Агент должен балансировать между предложением известных эффективных рекомендаций и исследованием новых вариантов.
- **Отложенная награда:** Действия, предпринятые на ранних этапах сессии, могут приводить к наградам позже (например, добавление в корзину может привести к покупке спустя некоторое время).
- **Контекстуальные особенности:** Учитывая характеристики товаров (цена, категория, бренд) и динамику сессии, состояние должно отражать как историю взаимодействия, так и текущий контекст.

---

### Итог

Формулировка задачи сводится к тому, чтобы на каждом шаге рекомендательной сессии агент, основываясь на состоянии, выбирал товар для рекомендации, и, используя обратную связь в виде наград (клики, добавления в корзину, покупки), обучался оптимальной политике, которая максимизирует общий доход или конверсию.
